### Building a Full Descriptor Library

In [ ]:
# descriptors.py — production-grade reusable descriptor set

import re
from typing import Any


class TypedField:
    """Base data descriptor with type enforcement."""

    expected_type: type = object

    def __set_name__(self, owner, name):
        self.name         = name
        self.storage_name = f"_desc_{name}"

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.storage_name, None)

    def __set__(self, obj, value):
        value = self.validate(value)
        setattr(obj, self.storage_name, value)

    def validate(self, value: Any) -> Any:
        """Override in subclasses for custom validation."""
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"{self.name!r} must be {self.expected_type.__name__}, "
                f"got {type(value).__name__}"
            )
        return value


class PositiveFloat(TypedField):
    expected_type = (int, float)

    def validate(self, value):
        super().validate(value)
        if value <= 0:
            raise ValueError(f"{self.name!r} must be positive, got {value}")
        return float(value)


class NonEmptyString(TypedField):
    expected_type = str

    def validate(self, value):
        super().validate(value)
        value = value.strip()
        if not value:
            raise ValueError(f"{self.name!r} cannot be empty")
        return value


class EmailField(TypedField):
    expected_type = str
    _pattern = re.compile(r"^[\w.+-]+@[\w-]+\.[a-zA-Z]{2,}$")

    def validate(self, value):
        super().validate(value)
        value = value.strip().lower()
        if not self._pattern.match(value):
            raise ValueError(f"{self.name!r} is not a valid email: {value!r}")
        return value


class BoundedInt(TypedField):
    expected_type = int

    def __init__(self, min_val=None, max_val=None):
        self.min_val = min_val
        self.max_val = max_val

    def validate(self, value):
        super().validate(value)
        if self.min_val is not None and value < self.min_val:
            raise ValueError(f"{self.name!r} must be >= {self.min_val}, got {value}")
        if self.max_val is not None and value > self.max_val:
            raise ValueError(f"{self.name!r} must be <= {self.max_val}, got {value}")
        return value